# Amazon Bedrock Flows 데모 — 이메일 에이전트 (모듈 3)

교재 모듈 3의 **사례 연구와 활동 1**을 Amazon Bedrock Flows로 만들어 실행합니다.

> AnyCompany Consulting의 고객 지원 부서는 시간당 1,000통 이상의 이메일을 받습니다. 이메일 에이전트는 다음을 해야 합니다.
> ① 스팸과 실제 이메일 분류 ② 간단한 질문 자동 응답 ③ 긴급하거나 복잡한 문제 에스컬레이션

활동 1의 정답 패턴을 그대로 흐름으로 옮깁니다.

| 태스크 | 패턴 | Flows 구현 |
|---|---|---|
| A. 스팸 분류 | 라우팅 | Prompt 노드(분류) → **Condition 노드** |
| B. 간단한 질문 자동 응답 | 라우팅 + **프롬프트 체인화** | 답장 초안 Prompt → 검토 Prompt |
| C. 긴급·복잡한 문제 에스컬레이션 | 라우팅 | 긴급: 티켓 작성 Prompt / 복잡: 담당자 메모 Prompt |

```
EmailInput ─▶ Classifier (Prompt) ─▶ RouteByCategory (Condition)
                                        ├─ SPAM    ─▶ SpamFiltered (Output)
                                        ├─ FAQ     ─▶ DraftReply (Prompt) ─▶ ReviewReply (Prompt) ─▶ AutoReply (Output)
                                        ├─ URGENT  ─▶ EscalationTicket (Prompt) ─▶ Escalated (Output)
                                        └─ default ─▶ TriageNote (Prompt) ─▶ HumanQueue (Output)
```

**만드는 리소스**: IAM 역할 1개와 흐름 1개(버전과 별칭 포함). 마지막 셀에서 모두 삭제합니다.
**사전 준비**: `pip install boto3`, Bedrock 모델 액세스(Claude Haiku 4.5), 권한 `bedrock:*Flow*`, `iam:CreateRole`, `iam:PutRolePolicy`

## 1. 설정

In [ ]:
import json, time
import boto3

REGION    = "us-west-2"
MODEL_ID  = "us.anthropic.claude-haiku-4-5-20251001-v1:0"   # 분류·답장 모두 빠르고 저렴한 모델
FLOW_NAME = "email-agent-flow-demo"
ROLE_NAME = "BedrockFlowsEmailAgentDemoRole"

account_id = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]
agent = boto3.client("bedrock-agent", region_name=REGION)            # 흐름 만들기
runtime = boto3.client("bedrock-agent-runtime", region_name=REGION)  # 흐름 실행하기
iam = boto3.client("iam")
print("계정:", account_id, "/ 리전:", REGION)

## 2. 수신함 — 교재 사례의 이메일 5통

In [ ]:
EMAILS = {
    "E1 스팸": "From: promo@lucky-winner.biz\nSubject: 축하합니다! 100만원 상품권 당첨\n\n"
              "24시간 안에 아래 링크를 눌러 카드 정보를 입력하시면 상품권을 지급합니다.",
    "E2 단순 문의": "From: doyoon.lee@example.com\nSubject: 반품 문의\n\n"
                 "노트북 거치대를 받았는데 생각보다 높이가 낮네요. 반품은 어떻게 하나요?",
    "E3 긴급": "From: doyoon.lee@example.com\nSubject: [긴급] 제가 하지 않은 결제\n\n"
              "방금 제 계정에서 제가 하지 않은 결제가 3건 발생했습니다. 오늘 안에 해결되지 않으면 소비자원에 신고하겠습니다.",
    "E4 복잡": "From: haneul.kim@example.com\nSubject: 주문한 제품이 도착하지 않았습니다\n\n"
              "주문한 무선 이어폰(ORD-1001)이 열흘째 도착하지 않았습니다. 환불을 받아야 하지만, 재고가 있다면 다시 주문하고 싶습니다.",
    "E5 후속": "From: seoyeon.park@example.com\nSubject: Re: 스마트워치 배송 문의\n\n"
              "지난번에 문의드린 스마트워치요, 아직 안 왔는데 지금 어디쯤 있나요?",
}
for k, v in EMAILS.items():
    print(f"[{k}] {v.splitlines()[1]}")

## 3. 흐름 실행 역할 (IAM)

흐름은 이 역할로 모델을 호출합니다. 신뢰 정책은 내 계정의 흐름만 역할을 맡을 수 있게 제한합니다.

In [ ]:
trust = {"Version": "2012-10-17", "Statement": [{
    "Effect": "Allow", "Principal": {"Service": "bedrock.amazonaws.com"}, "Action": "sts:AssumeRole",
    "Condition": {"StringEquals": {"aws:SourceAccount": account_id},
                  "ArnLike": {"aws:SourceArn": f"arn:aws:bedrock:{REGION}:{account_id}:flow/*"}}}]}
policy = {"Version": "2012-10-17", "Statement": [{
    "Effect": "Allow", "Action": ["bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream"],
    "Resource": ["arn:aws:bedrock:*::foundation-model/*", "arn:aws:bedrock:*:*:inference-profile/*"]}]}

try:
    role_arn = iam.create_role(RoleName=ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust))["Role"]["Arn"]
    created = True
except iam.exceptions.EntityAlreadyExistsException:
    role_arn = iam.get_role(RoleName=ROLE_NAME)["Role"]["Arn"]
    created = False
iam.put_role_policy(RoleName=ROLE_NAME, PolicyName="InvokeModel", PolicyDocument=json.dumps(policy))
if created:
    print("역할 생성 — 권한이 전파될 때까지 20초 대기")
    time.sleep(20)
print(role_arn)

## 4. 노드 정의

흐름은 **노드**와 **연결**로 이루어집니다. 반복되는 노드 형식은 작은 함수로 만듭니다.

- **Prompt 노드**: 프롬프트 템플릿 + 모델. `{{변수}}` 자리에 입력이 들어가고, 결과는 `modelCompletion`으로 나옵니다.
- **Output 노드**: 흐름의 결과. 어떤 Output 노드가 결과를 냈는지로 *어느 경로를 탔는지* 알 수 있습니다.
- **연결**: `Data`(값 전달)와 `Conditional`(Condition 노드에서 어느 갈래로 갈지)

In [ ]:
def prompt_node(name, template, variables, max_tokens=600):
    return {
        "name": name, "type": "Prompt",
        "inputs": [{"name": v, "type": "String", "expression": "$.data"} for v in variables],
        "outputs": [{"name": "modelCompletion", "type": "String"}],
        "configuration": {"prompt": {"sourceConfiguration": {"inline": {
            "modelId": MODEL_ID, "templateType": "TEXT",
            "templateConfiguration": {"text": {"text": template, "inputVariables": [{"name": v} for v in variables]}},
            "inferenceConfiguration": {"text": {"maxTokens": max_tokens, "temperature": 0.0}},
        }}}},
    }

def output_node(name):
    return {"name": name, "type": "Output",
            "inputs": [{"name": "document", "type": "String", "expression": "$.data"}],
            "configuration": {"output": {}}}

def data(source, target, target_input, source_output="modelCompletion"):
    return {"name": f"{source}_to_{target}_{target_input}", "source": source, "target": target, "type": "Data",
            "configuration": {"data": {"sourceOutput": source_output, "targetInput": target_input}}}

def conditional(target, condition):
    return {"name": f"Route_to_{target}", "source": "RouteByCategory", "target": target, "type": "Conditional",
            "configuration": {"conditional": {"condition": condition}}}

### 프롬프트

분류기는 Condition 노드가 비교할 수 있도록 **카테고리 이름 한 단어만** 출력하게 합니다.
FAQ 답장은 두 단계로 나눕니다(프롬프트 체인화). 검토 단계는 같은 FAQ와 초안을 대조해, FAQ와 다른 안내나 없는 약속을 고칩니다.

In [ ]:
CLASSIFIER = """고객 지원 이메일을 다음 중 하나로 분류하세요.
- SPAM: 광고, 피싱, 업무와 무관한 메일
- FAQ: 반품·교환·배송 기간·운영 시간처럼 절차나 정책을 묻는 일반 문의 ("교환할 수 있나요?"처럼 가능 여부만 묻는 경우 포함, 주문 번호 없음)
- URGENT: 보안 사고, 무단 결제, 법적 조치 언급 등 즉시 사람이 봐야 하는 건
- COMPLEX: 특정 주문의 조회·환불·재주문 처리 요청, 이전 문의의 후속 메일 등 그 밖의 모든 건

<email>
{{email}}
</email>

카테고리 이름 한 단어(SPAM, FAQ, URGENT, COMPLEX)만 출력하세요."""

FAQ = """<faq>
- 반품: 수령 후 14일 이내 마이페이지 > 주문내역 > 반품 신청. 택배 기사 방문 회수, 단순 변심 반품 배송비 3,000원
- 교환: 같은 상품을 다른 색상이나 다른 사이즈로 바꾸는 교환은 수령 후 14일 이내 1회 무료 (마이페이지 > 주문내역 > 교환 신청)
- 배송: 결제 후 1~3 영업일 내 출고, 출고 후 1~2일 내 도착
- 운영 시간: 평일 09:00~18:00, 주말·공휴일 휴무
</faq>"""

DRAFT = """AnyCompany 고객지원팀입니다. 아래 FAQ만 근거로 고객 이메일에 답장 초안을 쓰세요.

""" + FAQ + """

<email>
{{email}}
</email>"""

REVIEW = """당신은 고객 응대 품질 검토자입니다. 아래 FAQ와 대조해 답장 초안을 검토하고 최종 답장을 완성하세요.

""" + FAQ + """

- 초안이 FAQ와 다르게 안내한 부분은 FAQ에 맞게 고칩니다.
- FAQ에 없는 약속(보상, 쿠폰, 특정 날짜 보장 등)은 삭제합니다.
- 정중하고 간결한 4~6문장으로 다듬고, 끝에 "AnyCompany 고객지원팀 드림"을 붙입니다.
- 최종 답장 본문만 출력합니다.

<email>
{{email}}
</email>

<draft>
{{draft}}
</draft>"""

ESCALATION = """긴급 이메일입니다. 보안/결제 긴급 대응팀에 넘길 티켓을 작성하세요.
형식(이 형식 그대로, 한국어):
[우선순위] 긴급
[담당팀] 보안/결제 긴급 대응팀
[요약] 한 문장
[고객 요구] 항목별
[권장 즉시 조치] 항목별

<email>
{{email}}
</email>"""

TRIAGE = """사람 상담원이 처리할 이메일입니다. 담당자가 바로 일을 시작할 수 있도록 메모를 쓰세요.
형식(한국어):
[요청 요약] 한 문장
[확인할 정보] 주문 번호, 배송 상태, 재고, 환불 자격 등 필요한 조회 항목
[예상 처리] 단계별

<email>
{{email}}
</email>"""

### 흐름 정의 (노드 + 연결)

조건 연결은 *어느 갈래로 갈지만* 정합니다. 각 갈래의 노드가 받을 데이터(이메일 원문 등)는 Data 연결로 따로 이어 줍니다.

In [ ]:
nodes = [
    {"name": "EmailInput", "type": "Input", "outputs": [{"name": "document", "type": "String"}], "configuration": {"input": {}}},
    prompt_node("Classifier", CLASSIFIER, ["email"], max_tokens=10),
    {"name": "RouteByCategory", "type": "Condition",
     "inputs": [{"name": "category", "type": "String", "expression": "$.data"}],
     "configuration": {"condition": {"conditions": [
         {"name": "isSpam",   "expression": 'category == "SPAM"'},
         {"name": "isFaq",    "expression": 'category == "FAQ"'},
         {"name": "isUrgent", "expression": 'category == "URGENT"'},
         {"name": "default"},                       # COMPLEX, 또는 분류가 애매하면 사람에게 (안전 기본값)
     ]}}},
    prompt_node("DraftReply", DRAFT, ["email"]),
    prompt_node("ReviewReply", REVIEW, ["email", "draft"]),
    prompt_node("EscalationTicket", ESCALATION, ["email"]),
    prompt_node("TriageNote", TRIAGE, ["email"]),
    output_node("SpamFiltered"), output_node("AutoReply"), output_node("Escalated"), output_node("HumanQueue"),
]

connections = [
    # 분류 → 라우팅
    data("EmailInput", "Classifier", "email", source_output="document"),
    data("Classifier", "RouteByCategory", "category"),
    conditional("SpamFiltered", "isSpam"),
    conditional("DraftReply", "isFaq"),
    conditional("EscalationTicket", "isUrgent"),
    conditional("TriageNote", "default"),
    # 각 갈래에 이메일 원문 전달
    data("EmailInput", "DraftReply", "email", source_output="document"),
    data("EmailInput", "ReviewReply", "email", source_output="document"),
    data("EmailInput", "EscalationTicket", "email", source_output="document"),
    data("EmailInput", "TriageNote", "email", source_output="document"),
    # FAQ 갈래: 초안 → 검토 (프롬프트 체인화)
    data("DraftReply", "ReviewReply", "draft"),
    # 결과
    data("Classifier", "SpamFiltered", "document"),
    data("ReviewReply", "AutoReply", "document"),
    data("EscalationTicket", "Escalated", "document"),
    data("TriageNote", "HumanQueue", "document"),
]
definition = {"nodes": nodes, "connections": connections}
print(f"노드 {len(nodes)}개, 연결 {len(connections)}개")

## 5. 흐름 생성 → 준비(Prepare) → 버전 → 별칭

- **Prepare**: 정의를 검증하고 실행 가능한 상태(작업 초안)로 만듭니다.
- **버전**: 그 시점의 흐름을 변경 불가능한 스냅샷으로 고정합니다.
- **별칭**: 애플리케이션이 호출하는 이름. 별칭이 가리키는 버전만 바꾸면 코드 수정 없이 배포·롤백할 수 있습니다.

In [ ]:
existing = [f["id"] for f in agent.list_flows()["flowSummaries"] if f["name"] == FLOW_NAME]
args = dict(name=FLOW_NAME, executionRoleArn=role_arn, definition=definition,
            description="모듈 3 이메일 에이전트: 분류 → 라우팅 → 자동 응답/에스컬레이션")
if existing:
    flow_id = existing[0]
    agent.update_flow(flowIdentifier=flow_id, **args)
    print("흐름 갱신:", flow_id)
else:
    flow_id = agent.create_flow(**args)["id"]
    print("흐름 생성:", flow_id)

agent.prepare_flow(flowIdentifier=flow_id)
while (status := agent.get_flow(flowIdentifier=flow_id)["status"]) not in ("Prepared", "Failed"):
    time.sleep(3)
print("상태:", status)
if status == "Failed":
    print(json.dumps(agent.get_flow(flowIdentifier=flow_id).get("validations", []), ensure_ascii=False, indent=2))

In [ ]:
version = agent.create_flow_version(flowIdentifier=flow_id)["version"]
aliases = [a for a in agent.list_flow_aliases(flowIdentifier=flow_id)["flowAliasSummaries"] if a["name"] == "live"]
if aliases:
    alias_id = aliases[0]["id"]
    agent.update_flow_alias(flowIdentifier=flow_id, aliasIdentifier=alias_id, name="live",
                            routingConfiguration=[{"flowVersion": version}])
else:
    alias_id = agent.create_flow_alias(flowIdentifier=flow_id, name="live",
                                       routingConfiguration=[{"flowVersion": version}])["id"]
print(f"버전 {version} / 별칭 live ({alias_id})")
print(f"\n콘솔의 비주얼 빌더에서 노드 그래프 보기:\n"
      f"https://{REGION}.console.aws.amazon.com/bedrock/home?region={REGION}#/flows/{flow_id}")

> **데모 팁**: 위 콘솔 링크를 열어 두면 코드로 만든 흐름이 비주얼 빌더에 그대로 보입니다. 오른쪽 **테스트** 창에 이메일을 붙여 넣으면 실행 경로가 그래프 위에 표시됩니다.

## 6. 데모 — 흐름 실행

`InvokeFlow`는 결과를 스트림으로 돌려줍니다. `enableTrace=True`로 두면 노드마다 입력·출력 추적 이벤트가 함께 와서, **어떤 노드를 거쳤는지** 볼 수 있습니다.

In [ ]:
def run_flow(email_text, show_trace=True):
    start = time.time()
    resp = runtime.invoke_flow(
        flowIdentifier=flow_id, flowAliasIdentifier=alias_id, enableTrace=True,
        inputs=[{"nodeName": "EmailInput", "nodeOutputName": "document", "content": {"document": email_text}}])

    path, output_node, result, category = [], None, None, None
    for event in resp["responseStream"]:
        if "flowTraceEvent" in event:
            trace = event["flowTraceEvent"]["trace"]
            if "nodeOutputTrace" in trace:
                t = trace["nodeOutputTrace"]
                path.append(t["nodeName"])
                if t["nodeName"] == "Classifier":
                    category = t["fields"][0]["content"]["document"]
            elif "conditionNodeResultTrace" in trace:
                t = trace["conditionNodeResultTrace"]
                path.append(f"{t['nodeName']}→{t['satisfiedConditions'][0]['conditionName']}")
        elif "flowOutputEvent" in event:
            output_node = event["flowOutputEvent"]["nodeName"]
            result = event["flowOutputEvent"]["content"]["document"]
        elif "flowCompletionEvent" in event:
            completion = event["flowCompletionEvent"]["completionReason"]

    elapsed = time.time() - start
    if show_trace:
        print(f"분류: {category}   완료: {completion}   ({elapsed:.1f}초)")
        print("경로:", " → ".join(path + [output_node]))
        print("─" * 70)
        print(result)
    return {"category": category, "output_node": output_node, "seconds": round(elapsed, 1), "result": result}

### 6.1 단순 문의 — 라우팅 + 프롬프트 체인화 (태스크 B)

`Classifier → RouteByCategory(isFaq) → DraftReply → ReviewReply → AutoReply` 경로를 탑니다.

In [ ]:
_ = run_flow(EMAILS["E2 단순 문의"])

### 6.2 스팸 — 라우팅 (태스크 A)

답장 노드는 하나도 실행되지 않고 바로 `SpamFiltered`로 끝납니다. 모델 호출은 분류 1회뿐입니다.

In [ ]:
_ = run_flow(EMAILS["E1 스팸"])

### 6.3 긴급 — 에스컬레이션 (태스크 C)

In [ ]:
_ = run_flow(EMAILS["E3 긴급"])

### 6.4 복잡한 요청 — 기본 경로(사람 검토)

환불·재주문처럼 실제 시스템 조회와 실행이 필요한 요청은 이 흐름에서 처리하지 않고, 담당자 메모와 함께 사람에게 넘깁니다.

In [ ]:
_ = run_flow(EMAILS["E4 복잡"])

### 6.5 수신함 전체 처리

In [ ]:
rows = []
for name, text in EMAILS.items():
    r = run_flow(text, show_trace=False)
    rows.append({"email": name, **r})
    print(f"{name:10} | {r['category']:8} | {r['output_node']:13} | {r['seconds']:>4}초 | {r['result'][:45].replace(chr(10), ' ')}…")

### 직접 해 보기

아래 셀의 이메일을 바꿔 실행해 보세요. 예: 운영 시간 문의, 주문 번호가 있는 교환 요청, 욕설이 섞인 항의 메일.

분류가 애매한 메일은 `default` 갈래(사람 검토)로 갑니다. 잘못 분류되어도 고객에게 엉뚱한 자동 답장이 나가지 않도록 기본값을 가장 안전한 경로로 둔 것입니다.

In [ ]:
_ = run_flow("From: guest@example.com\nSubject: 교환 문의\n\n어제 받은 셔츠 사이즈가 작아요. 한 치수 큰 걸로 바꿀 수 있나요?")

## 7. 토론 — 이 흐름의 한계 (모듈 3 → 모듈 5로)

- 이 흐름은 **워크플로 에이전트**입니다. 경로가 미리 정해져 있어 빠르고 예측 가능하지만, E4처럼 주문 조회·환불·재주문이 필요한 건은 사람에게 넘길 수밖에 없습니다.
- 처리하려면? `TriageNote` 대신 **Lambda 노드**(주문 시스템 조회)나 **Agent 노드**(도구를 쓰는 Bedrock 에이전트)를 연결합니다.
- E5(후속 메일)는 이전 대화를 모르니 COMPLEX로 분류될 뿐입니다. 스레드 이력을 붙이려면 **S3 Retrieval / Knowledge Base 노드**나 메모리가 필요합니다.
- 콘텐츠 안전이 필요하면 Prompt 노드에 **Bedrock Guardrails**를 적용합니다.

## 8. 정리

흐름(별칭·버전 포함)과 IAM 역할을 삭제합니다.

In [ ]:
agent.delete_flow(flowIdentifier=flow_id, skipResourceInUseCheck=True)   # 별칭·버전도 함께 삭제
for p in iam.list_role_policies(RoleName=ROLE_NAME)["PolicyNames"]:
    iam.delete_role_policy(RoleName=ROLE_NAME, PolicyName=p)
iam.delete_role(RoleName=ROLE_NAME)
print("흐름과 IAM 역할을 삭제했습니다.")